# Latent space exploration

Loads a trained ArtGAN-GP checkpoint and explores its latent space:

1. **Random samples** &mdash; what the generator currently produces from `N(0, I)`.
2. **Slerp interpolation** &mdash; a smooth path between two seeds.
3. **Slerp grid** &mdash; a 2-D walk between four corner seeds.
4. **Local neighborhood walk** &mdash; small Gaussian perturbations of one seed to inspect local smoothness.

Set `CHECKPOINT_PATH` below to a `.pt` file written by `src/training/train.py`. The checkpoint must contain its embedded config (the train script saves it under `extra={'config': ...}`).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from omegaconf import OmegaConf

from src.models import build_models
from src.utils.checkpoint import load_checkpoint
from src.utils.interpolation import slerp_grid, slerp_path
from src.utils.visualize import denormalize, make_sample_grid

CHECKPOINT_PATH = Path('checkpoints/epoch_0100.pt')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
state = torch.load(str(CHECKPOINT_PATH), map_location=DEVICE)
cfg = OmegaConf.create(state['config'])

generator, critic = build_models(cfg)
generator.to(DEVICE)
critic.to(DEVICE)
load_checkpoint(CHECKPOINT_PATH, generator=generator, critic=critic, map_location=DEVICE)
generator.eval()

LATENT_DIM = cfg.model.latent_dim
print(f'Loaded checkpoint at epoch {state.get("epoch")} (latent_dim={LATENT_DIM}).')

## 1. Random samples

In [ ]:
def show_grid(grid_tensor, title=None, figsize=(8, 8)):
    fig, ax = plt.subplots(figsize=figsize)
    ax.imshow(grid_tensor.permute(1, 2, 0).cpu().numpy())
    ax.axis('off')
    if title:
        ax.set_title(title)
    plt.show()

with torch.no_grad():
    torch.manual_seed(0)
    z = torch.randn(64, LATENT_DIM, device=DEVICE)
    samples = generator(z)
show_grid(make_sample_grid(samples, nrow=8), title='Random samples')

## 2. Slerp interpolation between two seeds

Linear interpolation between two `N(0, I)` samples in high dimensions passes through the low-density interior of the latent ball &mdash; the generator was never trained on those inputs. **Spherical** linear interpolation (slerp) stays on the manifold and tends to produce smoother visual transitions, especially with WGAN-GP whose latent geometry is not regularized like a VAE's.

In [ ]:
def seeded_z(seed):
    g = torch.Generator(device=DEVICE)
    g.manual_seed(seed)
    return torch.randn(LATENT_DIM, device=DEVICE, generator=g)

z1 = seeded_z(0)
z2 = seeded_z(7)
path = slerp_path(z1, z2, n_steps=10)
with torch.no_grad():
    interp_imgs = generator(path)
show_grid(make_sample_grid(interp_imgs, nrow=10), title='Slerp interpolation (seed 0 -> seed 7)', figsize=(12, 2))

## 3. Slerp grid over four corner seeds

Bilinear-style: the top edge interpolates between corners 0 and 1, the bottom edge between corners 2 and 3, and each column slerps top &rarr; bottom. The four corners of the resulting grid match the four input seeds.

In [ ]:
corners = tuple(seeded_z(s) for s in (1, 2, 3, 4))
n_steps = 8
grid_z = slerp_grid(corners, n_steps=n_steps)
with torch.no_grad():
    grid_imgs = generator(grid_z)
show_grid(make_sample_grid(grid_imgs, nrow=n_steps), title=f'{n_steps}x{n_steps} slerp grid')

## 4. Local neighborhood walk

Small Gaussian perturbations of a single seed. If the generator is locally smooth, neighbours look like minor variations of the center image. If the latent space has cliffs, neighbours can be wildly different.


In [ ]:
torch.manual_seed(42)
z_center = seeded_z(0)
noise = torch.randn(63, LATENT_DIM, device=DEVICE) * 0.3
neighbours = torch.cat([z_center.unsqueeze(0), z_center.unsqueeze(0) + noise], dim=0)
with torch.no_grad():
    neigh_imgs = generator(neighbours)
show_grid(make_sample_grid(neigh_imgs, nrow=8), title='Center seed + 63 perturbations (sigma=0.3)')